# Scenario 1 — Hello-World through the Knowledge Graph

The **operation-coordination** interaction pattern. A hello-world middleware registers a workflow; a second middleware (a planner) *dispatches* an operation through the **event trigger** — creating it `queued` in the graph and ringing the hello resource's REST event trigger, resolving the peer purely through the graph — and the hello resource *pulls and runs* it, recording the outcome (ADR 0009/0010).

Self-contained: it clears and seeds a dedicated GraphDB repository (`GRAPHDB_*` env vars) and never touches production state.

In [1]:
import os
import threading
import time

import uvicorn
from graph_db_interface import GraphDB
from rdflib.namespace import RDF

from handlers import hello_world
from kapps_ogm import OGM
from kapps_semantic_middleware import SemanticMiddleware
from kapps_semantic_middleware.registration import mint_capability_iri, mint_workflow_iri
from kapps_semantic_middleware.vocabulary import CFC, OperationStatus, SVC

import seed

db = GraphDB.from_env()
print(f"Connected to GraphDB repository: {os.getenv('GRAPHDB_REPOSITORY')}")

INFO:KafkaManager:KafkaManager initialized


INFO:GraphDB:Using GraphDB repository 'Tests' as user 'etienneh'.


Connected to GraphDB repository: Tests


## Step 1 — Seed a Clean Repository

Clear the dedicated repository, load exactly the Scenario 1 ontology, and create the hello + planner resources.

In [2]:
seed.seed_scenario1(db)
print('Hello resource:  ', db.triple_exists((seed.HELLO_RESOURCE, RDF.type, seed.HELLO_RESOURCE_CLASS)))
print('Planner resource:', db.triple_exists((seed.PLANNER_RESOURCE, RDF.type, seed.PLANNER_RESOURCE_CLASS)))

Hello resource:   True
Planner resource: True


## Step 2 — Start the Hello-World Middleware

Wrap `hello_world` with `@workflow` and start the HTTP server. On startup the middleware registers its Service/Capability/Workflow structure and advertises a reachable endpoint. It also exposes the built-in `event_trigger` route.

In [3]:
mw1 = SemanticMiddleware(
    mode='resource', resource_iri=seed.HELLO_RESOURCE,
    service_class=seed.HELLO_SERVICE_CLASS, ogm=OGM(db=db),
    host='127.0.0.1', port=8993,
)
mw1.workflow(capability_class=seed.HELLO_CAPABILITY_CLASS,
             workflow_class=seed.HELLO_WORKFLOW_CLASS)(hello_world)

config = uvicorn.Config(mw1.app, host='127.0.0.1', port=8993, log_level='warning')
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
t0 = time.time()
while not server.started and time.time() - t0 < 30:
    time.sleep(0.05)
print('Hello middleware started on port 8993')
print('Routes:', [r.path for r in mw1.app.routes if 'workflows' in r.path])

Hello middleware started on port 8993
Routes: ['/workflows/event_trigger/execute', '/workflows/event_trigger/execute_background', '/workflows/event_trigger/description', '/workflows/event_trigger/interrupt', '/workflows/hello_world/execute', '/workflows/hello_world/execute_background', '/workflows/hello_world/description', '/workflows/hello_world/interrupt']


## Step 3 — Inspect What Registration Wrote

Only the instance-owned direction of each inverse is materialized (ADR 0008): a Service knows its resource (`isServiceOf`), a Workflow its Service (`isWorkflowOf`), a Capability its Workflow (`realizedByWorkflow`). The advertised address + endpoint are the reachability triples other agents discover.

In [4]:
# Per-instance since ADR 0022 — read off the instance rather than rebuilt from the resource.
service_iri = mw1.service_iri
cap_iri = mint_capability_iri(seed.HELLO_RESOURCE, 'hello_world')
wf_iri = mint_workflow_iri(service_iri, 'hello_world')
assert db.triple_exists((service_iri, SVC.isServiceOf, seed.HELLO_RESOURCE))
assert db.triple_exists((cap_iri, SVC.realizedByWorkflow, wf_iri))
assert db.triple_exists((wf_iri, SVC.isWorkflowOf, service_iri))
print('address: ', list(db.triples_get(sub=service_iri, pred=SVC.address)))
print('endpoint:', list(db.triples_get(sub=wf_iri, pred=SVC.endpoint)))

address:  [(IRI('https://example.org/kapps-demo#hello_resource_service'), IRI('https://w3id.org/circularfactory/Service#address'), 'http://127.0.0.1:8993')]
endpoint: [(IRI('https://example.org/kapps-demo#hello_resource_service_workflow_hello_world'), IRI('https://w3id.org/circularfactory/Service#endpoint'), 'http://127.0.0.1:8993/workflows/hello_world/execute')]


## Step 4 — Dispatch through the Event Trigger, then Pull-and-Run

The planner opens a `request(...)` transaction: on exit it creates the Operation `queued` (addressed via its Capability) and fires the hello resource's event trigger over REST. The hello resource then `claim_next()`s the queued Operation and runs the work; the context manager records the terminal status + provenance atomically.

In [5]:
planner = SemanticMiddleware(
    mode='resource', resource_iri=seed.PLANNER_RESOURCE,
    service_class=seed.PLANNER_SERVICE_CLASS, ogm=OGM(db=GraphDB.from_env()),
    host='127.0.0.1', port=8994,
)

with planner.request(capability_class=seed.HELLO_CAPABILITY_CLASS,
                     operation_class=str(CFC.Operation)) as op:
    pass  # helloworld takes no arguments
op_iri = op.iri
print('Planner dispatched operation:', op_iri)

with mw1.claim_next() as claimed:
    claimed.result = hello_world()
print('Hello resource pulled and ran it -> result:', repr(claimed.result))

INFO:KafkaManager:KafkaManager initialized


INFO:GraphDB:Using GraphDB repository 'Tests' as user 'etienneh'.


INFO:httpx:HTTP Request: POST http://127.0.0.1:8993/workflows/event_trigger/execute "HTTP/1.1 200 OK"


Planner dispatched operation: https://w3id.org/circularfactory/Core#Operation_op_34fe46174163


Hello resource pulled and ran it -> result: 'hello world'


## Step 5 — The Decision is Now Traceable in the Graph (R12)

The terminal transition wrote status `done` and the execution provenance in one atomic commit — the status *is* the provenance record (ADR 0009); there is no separate success boolean.

In [6]:
print('operationStatus:   ', list(db.triples_get(sub=op_iri, pred=SVC.operationStatus)))
print('executedByWorkflow:', list(db.triples_get(sub=op_iri, pred=SVC.executedByWorkflow)))
print('executionResult:   ', list(db.triples_get(sub=op_iri, pred=SVC.executionResult)))
assert list(db.triples_get(sub=op_iri, pred=SVC.operationStatus))[0][2] == OperationStatus.DONE

operationStatus:    [(IRI('https://w3id.org/circularfactory/Core#Operation_op_34fe46174163'), IRI('https://w3id.org/circularfactory/Service#operationStatus'), 'done')]
executedByWorkflow: [(IRI('https://w3id.org/circularfactory/Core#Operation_op_34fe46174163'), IRI('https://w3id.org/circularfactory/Service#executedByWorkflow'), IRI('https://example.org/kapps-demo#hello_resource_service_workflow_hello_world'))]
executionResult:    [(IRI('https://w3id.org/circularfactory/Core#Operation_op_34fe46174163'), IRI('https://w3id.org/circularfactory/Service#executionResult'), 'hello world')]


## Step 6 — Shutdown and Deregistration

On shutdown the middleware removes its reachability triples (address + endpoint) but preserves the individuals for provenance.

In [7]:
server.should_exit = True
thread.join(timeout=20)
time.sleep(0.5)
print('address removed: ', not list(db.triples_get(sub=service_iri, pred=SVC.address)))
print('endpoint removed:', not list(db.triples_get(sub=wf_iri, pred=SVC.endpoint)))
print('workflow individual preserved:', db.triple_exists((wf_iri, RDF.type, seed.HELLO_WORKFLOW_CLASS)))

address removed:  True
endpoint removed: True
workflow individual preserved: True
